# Final Project DS4420 - Eric Gerber 
Alina Gonzalez, Veronica Song, Sophia Yang 

Clean up and compile datasets from PetFinder dataset to match breed_id to breed, etc

Build out a CNN to identify key traits from animal given picture of pet
Build out MLP to predict adoption speed/ likelihood based off traits

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

In [2]:
# data is organized kinda weirdly, train.csv has all the columns in one but gotta link breed_id to actual breed name
petfinder = pd.read_csv("petfinder-adoption-prediction_data/train/train.csv").set_index("PetID")

petfinder.head()

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Sterilized,Health,Quantity,Fee,State,RescuerID,VideoAmt,Description,PhotoAmt,AdoptionSpeed
PetID,,,,,,,,,,,,,,,,,,,,,
86e1089a3,2,Nibble,3,299,0,1,1,7,0,1,...,2,1,1,100,41326,8480853f516546f6cf33aa88cd76c379,0,Nibble is a 3+ month old ball of cuteness. He ...,1.0,2
6296e909a,2,No Name Yet,1,265,0,1,1,2,0,2,...,3,1,1,0,41401,3082c7125d8fb66f7dd4bff4192c8b14,0,I just found it alone yesterday near my apartm...,2.0,0
3422e4906,1,Brisco,1,307,0,1,2,7,0,2,...,2,1,1,0,41326,fa90fa5b1ee11c86938398b60abc32cb,0,Their pregnant mother was dumped by her irresp...,7.0,3
5842f1ff5,1,Miko,4,307,0,2,1,2,0,2,...,2,1,1,150,41401,9238e4f44c71a75282e62f7136c6b240,0,"Good guard dog, very alert, active, obedience ...",8.0,2
850a43f90,1,Hunter,1,307,0,1,1,0,0,2,...,2,1,1,0,41326,95481e953f8aed9ec3d16fc4509537e8,0,This handsome yet cute boy is up for adoption....,3.0,2


# Building the CNN

In [10]:
""" 
Image data cleanup

Dataset provided images of animals and separate folder of jsons on description metadata
Match images of animals to their relevant metadata on dog breed(s) and color(s)

"""
# open the metadata and images provided by the dataset
TRAIN_METADATA_PATH = "petfinder-adoption-prediction_data/train_metadata"
TEST_METADATA_PATH = "petfinder-adoption-prediction_data/test_metadata"
TRAIN_IMAGES_PATH = "petfinder-adoption-prediction_data/train_images"
TEST_IMAGES_PATH = "petfinder-adoption-prediction_data/test_images"
ADOPTION_SPEED_TRAIN_CSV = "petfinder-adoption-prediction_data/train/train.csv"
ADOPTION_SPEED_TEST_CSV = "petfinder-adoption-prediction_data/test/test.csv"


# data set up is folder of images with ID in jpg name that corresponds to metadata json name in a folder of metadata..
# metadata json is weird too with multiple descriptions either saying "dog/ cat" or giving the actual breed
def extract_animal_type(labels):
    """
    from json metadata pull out if the animal iamge is a dog/ cat
    """
    descriptions = [l["description"].lower() for l in labels]
    if "dog" in descriptions:
        return 1
    elif "cat" in descriptions:
        return 0
    return None

# Need to extract breeds (1 and 2) from metadata too
# labels for breeds stored in two diff csv files

breeds = pd.read_csv("petfinder-adoption-prediction_data/breed_labels.csv")["BreedName"]
more_breeds = pd.read_csv("petfinder-adoption-prediction_data/BreedLabels.csv")["BreedName"]

# join the two dfs
breeds = pd.concat([breeds, more_breeds], ignore_index=True)

def extract_breeds(labels):
    """
    from json metadata pull out 2 breeds mentioned in the description
    """
    breeds = []
    for l in labels:
        desc = l["description"].lower()
        if desc in breeds:
            breeds.append(desc)
    return breeds[:2] 

# Need to extract colors from metadata json too
# colors are formatted like
#                 dominantColors: {     
#                     "color": {
#                         "red": 186,
#                         "green": 201,
#                         "blue": 229
#                     } with multiple "color" categories I fear and scores with each
# I think is giving RGB breakdown of the color identified and then score is amount of that color seen in the image
def extract_colors(color_data, top_k=3):
    """
    from json metadata pull out top colors of the animal based on scores provided
    """
    colors = sorted(color_data, key=lambda x: x["score"], reverse=True)
    top_colors = colors[:top_k]
    
    return [(c["color"]["red"], c["color"]["green"], c["color"]["blue"])
            for c in top_colors]

# Lowk, This is the real CNN code 

In [3]:
# running the functions to extract animal type, breed(s), and colors
import os
import json
from PIL import Image
from tqdm import tqdm

TRAIN_METADATA_PATH = "petfinder-adoption-prediction_data/train_metadata"
TEST_METADATA_PATH = "petfinder-adoption-prediction_data/test_metadata"
TRAIN_IMAGES_PATH = "petfinder-adoption-prediction_data/train_images"
TEST_IMAGES_PATH = "petfinder-adoption-prediction_data/test_images"
ADOPTION_SPEED_TRAIN_CSV = "petfinder-adoption-prediction_data/train/train.csv"
ADOPTION_SPEED_TEST_CSV = "petfinder-adoption-prediction_data/test/test.csv"


def load_data(IMAGES_PATH, ADOPTION_SPEED_PATH):
    """
    Open the image and metadata paths and load the data
    """
    data = []
    adoption_df = pd.read_csv(ADOPTION_SPEED_PATH)

    image_files = [f for f in os.listdir(IMAGES_PATH) if f.endswith(".jpg")]
    looked_at_pets = []
    # image_files = image_files[0:10] # FOR TESTING PURPOSES
    for img_file in tqdm(image_files):
        file_id = img_file.replace(".jpg", "")
        pet_id = file_id.split("-")[0]
        
        img_path = os.path.join(IMAGES_PATH, img_file)

        # skip already looked at pets, there are some dupe photos
        if pet_id in looked_at_pets: 
            continue

        pet_row = adoption_df[adoption_df['PetID'] == pet_id]

        # i don't think we want the pet row if empty 
        if pet_row.empty:
            continue  
        
        adoption_speed = pet_row.iloc[0]['AdoptionSpeed']

        image = Image.open(img_path).convert("RGB")
        data.append((image, adoption_speed))
        looked_at_pets.append(pet_id)

    return data

# converts all images into square shapes
def convert_image_to_scale(data, width=225, height=225):
    x_data = []
    y_data = [] 
    for img, speed in tqdm(data):
        resized = img.resize((width, height))

        arr = np.array(resized) / 255.0 
        x_data.append(arr)
        y_data.append(speed)

    x_data, y_data = np.array(x_data), np.array(y_data)

    X_train, X_test, y_train, y_test = train_test_split(
    x_data, y_data,
    test_size=0.2,
    random_state=42)
    return X_train, X_test, y_train, y_test
        
train_data = load_data(TRAIN_IMAGES_PATH, ADOPTION_SPEED_TRAIN_CSV)
X_train, X_test, y_train, y_test = convert_image_to_scale(train_data)


100%|██████████| 14652/14652 [01:16<00:00, 190.50it/s]


In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import Model, Input
from tensorflow.keras.losses import binary_crossentropy

inpx = Input(shape=(225, 225, 3))

con_layer = Conv2D(32, padding = 'same', kernel_size=(4, 4), strides=1, activation='relu')(inpx)
con_layer = MaxPooling2D((2,2))(con_layer)

con_layer2 = Conv2D(64, kernel_size=(3, 3), strides=1, activation='relu')(con_layer)

flat_G = Flatten()(con_layer2)

hid_layer = Dense(64, activation='relu')(flat_G)
hid_layer2 = Dense(32, activation='relu')(hid_layer)

out_layer = Dense(5, activation='sigmoid')(hid_layer2)

model = Model([inpx], out_layer)

model.compile(optimizer="adam",
              loss='categorical_crossentropy',
              metrics=['accuracy'])

from tensorflow.keras.utils import to_categorical

# Convert y_train and y_test to one-hot encoding
y_train_onehot = to_categorical(y_train, num_classes=5)
y_test_onehot = to_categorical(y_test, num_classes=5)

# Now train with one-hot encoded labels
history = model.fit(
    X_train, y_train_onehot,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

/Users/veronicasong/Documents/GitHub/ML2FinalProject/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch 1/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 150s 498ms/step - accuracy: 0.2656 - loss: 2.0137 - val_accuracy: 0.2614 - val_loss: 1.4796
Epoch 2/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 138s 470ms/step - accuracy: 0.2865 - loss: 1.4616 - val_accuracy: 0.2695 - val_loss: 1.4811
Epoch 3/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 129s 441ms/step - accuracy: 0.4602 - loss: 1.2591 - val_accuracy: 0.2733 - val_loss: 1.6924
Epoch 4/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 131s 446ms/step - accuracy: 0.7686 - loss: 0.6460 - val_accuracy: 0.2708 - val_loss: 2.5852
Epoch 5/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 136s 465ms/step - accuracy: 0.9442 - loss: 0.1961 - val_accuracy: 0.2670 - val_loss: 4.3834
Epoch 6/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 132s 451ms/step - accuracy: 0.9839 - loss: 0.0808 - val_accuracy: 0.2665 - val_loss: 5.0362
Epoch 7/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 134s 459ms/step - accuracy: 0.9902 - loss: 0.0343 - val_accuracy: 0.2644 - val_loss: 6.0176
Epoch 8/30
293/293 ━━━━━━━━━━━━━━━━━━━━ 137s 466ms/step - accuracy: 0.9920 -

In [5]:
model.save('pet_adoption_model.h5')


In [11]:
y_pred = model.predict(X_test)

92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 86ms/step


In [12]:
y_pred = np.argmax(y_pred, axis=1)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 25.49%

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        80
           1       0.23      0.24      0.24       609
           2       0.28      0.21      0.24       805
           3       0.26      0.30      0.28       669
           4       0.25      0.29      0.27       768

    accuracy                           0.25      2931
   macro avg       0.21      0.21      0.21      2931
weighted avg       0.25      0.25      0.25      2931


Confusion Matrix:
[[  0  17  16  15  32]
 [  3 147 126 145 188]
 [  4 164 173 206 258]
 [  3 136 128 203 199]
 [  4 169 171 200 224]]


Alina code ends

In [8]:
def predict_adoption_speed(image_path):
    """Predict adoption speed for a new image"""
    img = Image.open(image_path).convert('RGB')
    img = img.resize((225, 225))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    
    pred = model.predict(img_array, verbose=0)
    
    if n_classes <= 10:
        speed_class = np.argmax(pred)
        speed_label = le.inverse_transform([speed_class])[0]
        confidence = pred[0][speed_class]
        print(f"Predicted speed: {speed_label} (confidence: {confidence:.2%})")
    else:
        speed_days = pred[0][0]
        print(f"Predicted adoption time: {speed_days:.1f} days")
    
    return pred

In [9]:
def process_sample(image, meta):
    """
    from each file, extract the key visual components of the pet (breed(s) and colors)
    """
    labels = meta["labelAnnotations"]
    colors = meta["imagePropertiesAnnotation"]["dominantColors"]["colors"]

    animal = extract_animal_type(labels)
    breeds = extract_breeds(labels)
    color_vals = extract_colors(colors)

    return image, animal, breeds, color_vals

In [14]:
%pip install torch

  Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl.metadata (30 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.2.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl (73.6 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.19.1-py3-none-any.whl (15 kB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [torch]32m6/7 [torch]]x]
Note: you may need to restart the kernel to use updated packages.


In [15]:
# use torchvision to transform the images
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def prepare_batch(data_batch):
    """
    separate out images, animal type, breeds, and colors into separate lists for batching
    """
    images, animals, breeds, colors = [], [], [], []

    for img, meta in tqdm(data_batch):
        img, animal, breed, color = process_sample(img, meta)

        images.append(transform(img))
        animals.append(animal)
        breeds.append(breed)
        colors.append(color)

    return images, animals, breeds, colors

def create_batches(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

ModuleNotFoundError: No module named 'torchvision'

## THIS IS WHERE HISTORIC HOMEWORK CODE STARTS THAT I WAS TRYING TO REFERENCE

In [10]:
# setting all images to same img size
from PIL import Image, ImageOps

def convert_image_to_scale_with_padding(image_path, target_width, target_height):
    """
    Converts a single .jpg image to the target dimensions.

    Args:
        image_path (str): Path to the input .jpg image.
        target_width (int, optional): Target width for padding
        target_height (int, optional): Target height for padding

    Returns:
        np.ndarray: image as a numpy array, with padding applied.
    """
    # Open the image
    with Image.open(image_path) as img:
        # If target dimensions are provided, pad the image
        if target_width and target_height:
            padded_img = ImageOps.pad(img, (target_width, target_height), color=0)
        else:
            print("Need to provide target width and height")

        # Convert the image to a numpy array
        array = np.array(padded_img)

    return array

In [11]:
img_rows, img_cols = 540, 499

# define the input with the image rows, cols, and grayscale column
inpx = Input(shape=(img_rows, img_cols, 1))

# define convolutional layer 1
con_layer = Conv2D(1, padding = 'same', kernel_size=(4, 4), strides=1, activation=None)(inpx)
con_layer = MaxPooling2D((2,2))(con_layer)
# define convolutional layer 2
con_layer2 = Conv2D(1, kernel_size=(3, 3), strides=1, activation=None)(con_layer)

# flattening the image
flat_G = Flatten()(con_layer2)

# establishing hidden layers
# hidden layer 1
hid_layer = Dense(64, activation='relu')(flat_G)
# hidden layer 2
hid_layer2 = Dense(32, activation='relu')(hid_layer)

# defining output function (5 outputs for 5 traits of pet we want to identify)
out_layer = Dense(5, activation='sigmoid')(hid_layer2)

# fitting the model
model = Model([inpx], out_layer)

model.compile(optimizer="adam",
              loss=binary_crossentropy,
              metrics=['accuracy'])

In [12]:
# putting all of it togehter now 
import pickle

# PICKLE SO I DONT HAVE TO RUN EVERY TIME :w;
data = load_data(TRAIN_IMAGES_PATH, TRAIN_METADATA_PATH)

with open('load_data_obj.pickle', 'wb') as handle:
    pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)

100%|██████████| 58311/58311 [10:55<00:00, 88.99it/s]  


In [28]:
test_data = load_data(TRAIN_IMAGES_PATH, ADOPTION_SPEED_TRAIN_CSV)

100%|██████████| 10000/10000 [00:04<00:00, 2088.22it/s]


In [ ]:
# If you would like to unpickle ;3

# import pickle

# with open('load_data_obj.pickle', 'rb') as handle:
#     b = pickle.load(handle)

EOFError: Ran out of input

In [ ]:
result = prepare_batch(data[1:100])

images = []
result = []

100%|██████████| 99/99 [00:00<00:00, 272.09it/s]


[tensor([[[0.2353, 0.2353, 0.2510,  ..., 0.0549, 0.0588, 0.0588],
          [0.2353, 0.2392, 0.2549,  ..., 0.0549, 0.0588, 0.0588],
          [0.2392, 0.2471, 0.2627,  ..., 0.0549, 0.0588, 0.0588],
          ...,
          [0.0510, 0.0510, 0.0510,  ..., 0.2275, 0.2039, 0.1843],
          [0.0510, 0.0510, 0.0510,  ..., 0.2353, 0.1843, 0.1647],
          [0.0510, 0.0510, 0.0510,  ..., 0.1843, 0.1843, 0.1647]],
 
         [[0.2588, 0.2627, 0.2745,  ..., 0.0549, 0.0588, 0.0588],
          [0.2588, 0.2627, 0.2784,  ..., 0.0549, 0.0588, 0.0588],
          [0.2549, 0.2627, 0.2784,  ..., 0.0549, 0.0588, 0.0588],
          ...,
          [0.0510, 0.0510, 0.0510,  ..., 0.3961, 0.3725, 0.3529],
          [0.0510, 0.0510, 0.0510,  ..., 0.3961, 0.3490, 0.3255],
          [0.0510, 0.0510, 0.0510,  ..., 0.3373, 0.3373, 0.3176]],
 
         [[0.2039, 0.2039, 0.2118,  ..., 0.0471, 0.0510, 0.0510],
          [0.2039, 0.2039, 0.2157,  ..., 0.0471, 0.0510, 0.0510],
          [0.2000, 0.2039, 0.2196,  ...,

In [21]:
model.fit(result, y_train, epochs=10)

NameError: name 'y_train' is not defined

In [ ]:
# calculate the accuracy score
score = model.evaluate(X_test, y_test, verbose=0)
print('loss=', score[0])
print('accuracy=', score[1])